# Cross-Market FIFA Prediction Arbitrage Scanner

This notebook is the research control panel for the MVP pivot: compare equivalent FIFA / World Cup prediction markets on Polymarket and Kalshi, then flag cases where buying complementary outcomes costs less than 1 dollar after conservative buffers.

The notebook is alerts-only. It does not trade, manage keys, use WebSockets, or auto-approve mappings.

## MVP Rules

The scanner only alerts when all of these are true:

- The market pair is manually approved in `config/fifa_market_mappings.csv`.
- Draw handling, extra time handling, penalties handling, and settlement notes are filled in.
- Both live asks are available.
- Both legs clear the minimum displayed depth.
- The buffered complementary cost leaves enough edge.

The two directions are:

```text
buy Polymarket YES + buy Kalshi NO
buy Kalshi YES + buy Polymarket NO
```

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import httpx
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = Path('/Users/qisongqiao/Warehouse/cv/project_simulation/prediction_market')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from prediction_market.fifa_arbitrage import (
    MAPPING_COLUMNS,
    approved_mappings,
    build_approval_candidates,
    normalize_fifa_candidates,
    run_fifa_snapshot,
    score_cross_market_arbitrage,
    suggest_manual_mappings,
    validate_manual_mappings,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'fifa_arbitrage'
MAPPING_PATH = PROJECT_ROOT / 'config' / 'fifa_market_mappings.csv'
RUN_LIVE_FIFA_NOTEBOOK = os.getenv('RUN_LIVE_FIFA_NOTEBOOK') == '1'

print(f'Project root: {PROJECT_ROOT}')
print(f'Mapping path: {MAPPING_PATH}')
print(f'Live API mode: {RUN_LIVE_FIFA_NOTEBOOK}')

## Manual Mapping Template

This file is intentionally the gatekeeper. Candidate discovery may find many related contracts, but alerts only run for rows marked `approved` after manual settlement review.

In [ ]:
mapping_template = pd.read_csv(MAPPING_PATH, dtype=str, keep_default_na=False)
print(f'Mapping rows: {len(mapping_template)}')
display(mapping_template.head())
print('Required columns:')
print(', '.join(MAPPING_COLUMNS))

## Fixture Candidate Discovery

The fixture below mirrors the normalized schema used by live discovery. It gives the notebook something meaningful to run even when there is no current Kalshi FIFA equivalent online.

In [ ]:
fixture_polymarket = [
    {
        'id': 'pm-market',
        'conditionId': '0xfifa',
        'slug': 'usa-france-world-cup',
        'question': 'USA vs France World Cup: USA to win?',
        'active': True,
        'closed': False,
        'outcomes': '["USA", "France or draw"]',
        'clobTokenIds': '["pm-yes", "pm-no"]',
        'description': '2026 FIFA World Cup match winner in regular time.',
    }
]
fixture_kalshi = [
    {
        'ticker': 'KXWCFINAL-26USA',
        'title': 'World Cup: USA to win?',
        'subtitle': 'USA vs France',
        'category': 'Sports',
        'status': 'active',
        'close_time': '2026-07-01T00:00:00Z',
        'yes_sub_title': 'USA wins',
        'no_sub_title': 'USA does not win',
        'rules_primary': 'Market resolves Yes if USA wins this World Cup match in regulation.',
    }
]

candidate_frame = normalize_fifa_candidates(
    fixture_polymarket,
    fixture_kalshi,
    run_id='notebook-fixture',
    retrieved_at='2026-06-21T00:00:00Z',
)
display(candidate_frame[['venue', 'market_id', 'ticker_or_slug', 'title', 'outcomes', 'keyword_hits', 'rules_text']])

## Approval Workbench

The scanner now builds two review tables before any manual approval: `approval_candidates` gives one enriched row per venue market, and `suggested_mappings` proposes likely pairs while keeping them `review_required`. These are the files to inspect when `config/fifa_market_mappings.csv` is still empty.


In [ ]:
approval_frame = build_approval_candidates(candidate_frame)
suggested_mapping_frame = suggest_manual_mappings(approval_frame, min_score=70)
display(approval_frame[['venue', 'market_type', 'event_title', 'event_date', 'event_match_key', 'outcome_label', 'title', 'settlement_summary', 'approval_notes']])
display(suggested_mapping_frame[['suggestion_status', 'match_score', 'market_type', 'event_name', 'proposition', 'outcome_label', 'polymarket_slug', 'kalshi_ticker', 'draw_handling', 'extra_time_handling', 'penalties_handling']])


## Manual Approval Example

This example shows the level of settlement detail required before an alert can fire. The draw/extra-time/penalty fields are not decoration; they are the main risk control for football markets.

In [ ]:
fixture_mapping = pd.DataFrame(
    [
        {
            'mapping_id': 'fixture-usa-france',
            'status': 'approved',
            'event_name': 'USA vs France',
            'proposition': 'USA to win in regular time',
            'polymarket_market_id': '0xfifa',
            'polymarket_slug': 'usa-france-world-cup',
            'polymarket_yes_token_id': 'pm-yes',
            'polymarket_no_token_id': 'pm-no',
            'polymarket_yes_outcome': 'USA',
            'polymarket_no_outcome': 'France or draw',
            'kalshi_ticker': 'KXWCFINAL-26USA',
            'draw_handling': 'draw means NO',
            'extra_time_handling': 'extra time excluded',
            'penalties_handling': 'penalties excluded',
            'settlement_notes': 'Both fixture markets settle on regular-time USA win.',
            'reviewer': 'notebook-fixture',
            'reviewed_at': '2026-06-21T00:00:00Z',
            'notes': 'Synthetic fixture used to demonstrate the scanner math.',
        }
    ],
    columns=MAPPING_COLUMNS,
)
validated_mapping = validate_manual_mappings(fixture_mapping)
eligible_mapping = approved_mappings(fixture_mapping)
display(validated_mapping[['mapping_id', 'status', 'is_approved', 'validation_errors', 'draw_handling', 'extra_time_handling', 'penalties_handling']])

## Fixture Price Comparison

Kalshi orderbooks expose YES and NO bids. The scanner derives asks from the opposite side:

```text
yes_ask = 1 - best_no_bid
no_ask = 1 - best_yes_bid
```

The fixture table is already normalized to the scanner output schema.

In [ ]:
orderbook_frame = pd.DataFrame(
    [
        {
            'run_id': 'notebook-fixture',
            'retrieved_at': '2026-06-21T00:00:00Z',
            'mapping_id': 'fixture-usa-france',
            'venue': 'polymarket',
            'market_id': '0xfifa',
            'yes_bid': 0.37,
            'yes_ask': 0.40,
            'no_bid': 0.56,
            'no_ask': 0.70,
            'yes_bid_depth': 20.0,
            'yes_ask_depth': 25.0,
            'no_bid_depth': 20.0,
            'no_ask_depth': 25.0,
            'raw_orderbook': '{}',
            'error': '',
        },
        {
            'run_id': 'notebook-fixture',
            'retrieved_at': '2026-06-21T00:00:00Z',
            'mapping_id': 'fixture-usa-france',
            'venue': 'kalshi',
            'market_id': 'KXWCFINAL-26USA',
            'yes_bid': 0.48,
            'yes_ask': 0.44,
            'no_bid': 0.56,
            'no_ask': 0.52,
            'yes_bid_depth': 40.0,
            'yes_ask_depth': 30.0,
            'no_bid_depth': 30.0,
            'no_ask_depth': 40.0,
            'raw_orderbook': '{}',
            'error': '',
        },
    ]
)
display(orderbook_frame[['mapping_id', 'venue', 'yes_bid', 'yes_ask', 'no_bid', 'no_ask', 'yes_ask_depth', 'no_ask_depth']])

## Alert Scoring

Defaults are conservative: 1 cent slippage total across two legs, 1 cent fee buffer, 2 cent minimum net edge, and 10 contracts minimum displayed depth per leg.

In [ ]:
alerts = score_cross_market_arbitrage(
    eligible_mapping,
    orderbook_frame,
    run_id='notebook-fixture',
    min_net_edge=0.02,
    slippage_buffer_per_leg=0.005,
    fee_buffer_total=0.01,
    min_depth_per_leg=10,
    detected_at='2026-06-21T00:00:00Z',
)
display(alerts[['is_alert', 'direction', 'gross_cost', 'buffered_cost', 'net_edge', 'min_depth', 'exclusion_reason']])

## Historical Logs

The actual scanner writes appendable local artifacts under `data/fifa_arbitrage/`. Fixture outputs are written here too so downstream backtesting code can read the same shapes.

In [ ]:
processed_dir = OUTPUT_DIR / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
candidate_frame.to_parquet(processed_dir / 'venue_market_candidates_fixture.parquet', index=False)
approval_frame.to_parquet(processed_dir / 'approval_candidates_fixture.parquet', index=False)
suggested_mapping_frame.to_parquet(processed_dir / 'suggested_mappings_fixture.parquet', index=False)
validated_mapping.to_parquet(processed_dir / 'manual_mappings_snapshot_fixture.parquet', index=False)
orderbook_frame.to_parquet(processed_dir / 'orderbook_snapshots_fixture.parquet', index=False)
alerts.to_parquet(processed_dir / 'arbitrage_alerts_fixture.parquet', index=False)
print(f'Wrote fixture tables to {processed_dir}')

## Optional Live Snapshot

Live mode calls public Polymarket/Kalshi REST APIs and uses the real manual mapping file. If there are no approved mappings yet, it still records a valid no-alert run.

```bash
RUN_LIVE_FIFA_NOTEBOOK=1 jupyter notebook notebooks/05_cross_market_fifa_arbitrage_scanner.ipynb
```

In [ ]:
if RUN_LIVE_FIFA_NOTEBOOK:
    result = run_fifa_snapshot(output_dir=OUTPUT_DIR, mapping_path=MAPPING_PATH, market_limit=1000, page_size=200)
    for name, frame in result['tables'].items():
        print(name, frame.shape)
    display(result['tables']['approval_candidates'][['venue', 'market_type', 'event_title', 'event_date', 'event_match_key', 'outcome_label', 'ticker_or_slug', 'title']].head(50))
    display(result['tables']['suggested_mappings'][['suggestion_status', 'match_score', 'event_name', 'proposition', 'outcome_label', 'polymarket_slug', 'kalshi_ticker']].head(50))
    display(result['tables']['arbitrage_alerts'].head(20))
else:
    print('Skipping live snapshot. Set RUN_LIVE_FIFA_NOTEBOOK=1 to call live APIs.')


## Scheduler Command

The local watcher is the MVP orchestrator. It runs the same snapshot every 60 seconds by default and appends each run.

```bash
poly-x-kalshi-fifa-watch --interval-seconds 60
```

For a quick dry exercise without waiting forever, use:

```bash
poly-x-kalshi-fifa-watch --max-ticks 2 --interval-seconds 5
```

## Feasibility Verdict

For a real run, read the verdict from the alert history:

- If approved mappings are empty, the project is still in mapping/bootstrap mode.
- If alerts appear but disappear quickly, the next question is executable persistence.
- If no alerts appear across repeated snapshots, the first answer may be that FIFA cross-market gaps are rare or mapping coverage is too thin.

In [ ]:
alert_count = int(alerts['is_alert'].sum()) if not alerts.empty else 0
if alert_count:
    verdict = 'fixture viable: alert math and logging work; live viability still depends on approved real mappings.'
elif eligible_mapping.empty:
    verdict = 'mapping bootstrap: add approved mappings before judging live opportunity frequency.'
else:
    verdict = 'no fixture alert under current thresholds.'
print(verdict)